In [1]:
# %pip install lightning

In [ ]:
import torch

from torchvision.datasets import EuroSAT
from torchvision.transforms import ToTensor
from torch.utils.data import random_split, DataLoader
import torch.nn as nn

from euroshap.utils.utils import get_device
from euroshap.modeling.model import SatResNet, LightningClassifier

device = get_device()
torch.set_float32_matmul_precision('medium')

data = EuroSAT(root='../../data/processed', transform=ToTensor(), download=False)
N = len(data)
train_size = int(0.8 * N)
temp_size = N - train_size  # remaining 20%
test_size = int(temp_size / 2)
val_size = temp_size - test_size  # this ensures the sum equals N

train, temp = random_split(data, [train_size, temp_size])
test, val = random_split(temp, [test_size, val_size])

batch_size = 32
train_loader = DataLoader(
    train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=15,
    persistent_workers=True
)
val_loader = DataLoader(
    val,
    batch_size=batch_size,
    shuffle=False,
    num_workers=15,
    persistent_workers=True
)
test_loader = DataLoader(
    test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=15,
    persistent_workers=True
)

resnet = SatResNet(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.parameters(), lr=0.001)

lit_model = LightningClassifier(model=resnet, lr=0.001)

Using cuda device


In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
import pytorch_lightning as pl

checkpoint_callback = ModelCheckpoint(
    dirpath='../../models/',
    filename='satresnet_{epoch}',
    save_top_k=-1,
    every_n_epochs=1
)

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=[checkpoint_callback],
    accelerator='auto'
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [6]:
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | SatResNet        | 11.2 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
70        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
